In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

import sys
sys.path.append("../../utils/")

from utils import *

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_DATASET_RESULTADO = "CIC17__cleanning__v1"

# Ajusta esto si hace falta
RUTA_BASE_RAW = PROJECT_ROOT / "02_datasets" / "raw" / "CIC17"
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed" / NOMBRE_DATASET_RESULTADO

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_RESULTADO}.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
CORR_THRESHOLD = 0.95
MIN_CLASS_IMPUTE = 50

# ===== COLUMNAS A ELIMINAR MANUALMENTE SI EXISTEN =====
COLUMNAS_A_ELIMINAR = [
    "FLOW_ID",
    "SRC_IP",
    "SRC_PORT",
    "DST_IP"
]

In [ ]:
print("Carpeta actual del notebook:")
print(PROJECT_ROOT)
print()
print("Ruta base raw:")
print(Path(RUTA_BASE_RAW).resolve())
print()
print("Ruta salida:")
print(Path(RUTA_SALIDA).resolve())
print()

In [ ]:
df = cargar_dataset(
    ruta_base=RUTA_BASE_RAW
)

shape_original = df.shape

print("Forma original del dataset:")
print(shape_original)

df.head()

In [ ]:
df = homogeneizar_columnas(df)

print("Primeras columnas tras homogeneización:")
print(df.columns.tolist()[:20])

In [ ]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución inicial de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
columnas_presentes_para_eliminar = [c for c in COLUMNAS_A_ELIMINAR if c in df.columns]

df = eliminar_columnas(df, columnas_presentes_para_eliminar)

print("Columnas eliminadas manualmente:")
print(columnas_presentes_para_eliminar)
print()
print("Forma actual:")
print(df.shape)

In [ ]:
df = limpiar_infinitos_y_vacios(df)

print("Limpieza de infinitos y vacíos completada.")

In [ ]:
filas_antes_duplicados = len(df)

df = eliminar_filas_duplicadas(df)

filas_despues_duplicados = len(df)
duplicados_eliminados = filas_antes_duplicados - filas_despues_duplicados

print("Duplicados eliminados:", duplicados_eliminados)
print("Forma actual:", df.shape)

In [ ]:
df_sin_nulos, df_con_nulos = separar_filas_con_y_sin_nulos(df)

print("Filas sin nulos:", len(df_sin_nulos))
print("Filas con nulos:", len(df_con_nulos))

In [ ]:
df, filas_imputadas, filas_eliminadas_nulos = imputar_o_eliminar_nulos_por_clase(
    df_sin_nulos=df_sin_nulos,
    df_con_nulos=df_con_nulos,
    label_col=LABEL_COL,
    min_class_impute=MIN_CLASS_IMPUTE
)

print("Filas imputadas:", filas_imputadas)
print("Filas eliminadas por nulos:", filas_eliminadas_nulos)
print("Forma actual:", df.shape)

In [ ]:
columnas_antes_constantes = df.shape[1]

df = eliminar_columnas_constantes(df)

columnas_despues_constantes = df.shape[1]
constantes_eliminadas = columnas_antes_constantes - columnas_despues_constantes

print("Columnas constantes eliminadas:", constantes_eliminadas)
print("Forma actual:", df.shape)

In [ ]:
df, columnas_categoricas_codificadas = codificar_columnas_categoricas_one_hot(
    df,
    label_col=LABEL_COL
)

print("Columnas categóricas originales codificadas:")
print(columnas_categoricas_codificadas)
print()
print("Forma actual:", df.shape)

In [ ]:
columnas_antes_corr = df.shape[1]

df = eliminar_columnas_altamente_correlacionadas(
    df,
    threshold=CORR_THRESHOLD,
    label_col=LABEL_COL
)

columnas_despues_corr = df.shape[1]
corr_eliminadas = columnas_antes_corr - columnas_despues_corr

print("Columnas eliminadas por alta correlación:", corr_eliminadas)
print("Forma final tras limpieza:", df.shape)

In [ ]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
df = df[feature_cols + [LABEL_COL]]

print("Última columna:", df.columns[-1])
df.head()

In [ ]:
df, mapping = codificar_etiqueta_label(df, LABEL_COL)
mapping

In [ ]:
print("Distribución final de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
guardar_dataset_csv(
    df=df,
    nombre_archivo=NOMBRE_DATASET_LIMPIO,
    ruta=RUTA_SALIDA
)

print("Dataset limpio guardado correctamente.")
print(Path(RUTA_SALIDA).resolve() / NOMBRE_DATASET_LIMPIO)